# STOCKER — Full GPU Training Pipeline

**S&P 500 (503 sembol) × 5 yıl × 8 horizon × FinBERT sentiment × 4 model**

| Aşama | Süre (T4) |
|--------|----------|
| 1. Veri toplama (500 sembol daily + intraday) | ~10 dk |
| 2. Finnhub haber toplama (top 100) | ~30 dk |
| 3. Dataset build (125 feature, 8 horizon) | ~30-60 dk |
| 4. Model eğitimi (4 model + meta-learner) | ~1-2 saat |

**Runtime > Change runtime type > GPU (T4)** seçili olmalı!

## 0. Config

In [ ]:
# ── Config ───────────────────────────────────────────────────
GITHUB_REPO = "beratkaskaloglu/stocker"
BRANCH = "main"
MARKET = "US"
EPOCHS = 100
BATCH_SIZE = 64
SAVE_TO_DRIVE = True

# Finnhub API key (haber toplama icin)
FINNHUB_API_KEY = "d73s199r01qjjol44ul0d73s199r01qjjol44ulg"

# Kac sembol icin haber toplansin (0=haber toplama, 100=top 100, 500=hepsi)
NEWS_TOP_N = 100

# Intraday veri de toplansin mi (1h/4h horizon icin)
COLLECT_INTRADAY = True

## 1. GPU Kontrol & Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("GPU yok! Runtime > Change runtime type > GPU secin.")

## 2. Repo Clone & Dependencies

In [ ]:
import os, sys
if not os.path.exists("/content/stocker"):
    !git clone --depth 1 -b {BRANCH} https://github.com/{GITHUB_REPO}.git /content/stocker
else:
    !cd /content/stocker && git pull origin {BRANCH}
os.chdir("/content/stocker")
# Ensure project root is on sys.path for all 'from core.*' imports
if "/content/stocker" not in sys.path:
    sys.path.insert(0, "/content/stocker")
print(f"CWD: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

In [ ]:
!pip install -q loguru ta PyWavelets pyts statsmodels finnhub-python click pyyaml python-dotenv tqdm 2>&1 | tail -3
print("Dependencies installed.")

In [ ]:
# .env olustur (Finnhub key)
with open("/content/stocker/.env", "w") as f:
    f.write(f"FINNHUB_API_KEY={FINNHUB_API_KEY}\n")
    f.write("DEVICE=cuda\n")
os.environ["FINNHUB_API_KEY"] = FINNHUB_API_KEY
print(".env created")

## 3. Google Drive Mount

In [ ]:
DRIVE_DIR = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/stocker"
    os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
    print(f"Drive: {DRIVE_DIR}")

## 4. Veri Toplama (S&P 500 × 503 sembol)

| Veri | Kaynak | Kapsam |
|------|--------|--------|
| Daily OHLCV | yfinance | 503 sembol × 5 yıl |
| Intraday 1h | yfinance | 503 sembol × 2 yıl |
| Haberler | Finnhub | Top 100 sembol × 5 yıl |

In [ ]:
import warnings, shutil, time
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")

os.makedirs("/content/stocker/data", exist_ok=True)

DATASET_DIR = "/content/stocker/data/sp500_dataset"
OHLCV_CSV = "/content/stocker/data/ohlcv_sp500.csv"
INTRADAY_CSV = "/content/stocker/data/intraday_1h_sp500.csv"
NEWS_JSON = "/content/stocker/data/news_sp500.json"

# Drive'dan varsa kopyala
SKIP_DAILY = False
SKIP_INTRADAY = False
SKIP_NEWS = False

if DRIVE_DIR:
    if Path(f"{DRIVE_DIR}/data/ohlcv_sp500.csv").exists():
        shutil.copy2(f"{DRIVE_DIR}/data/ohlcv_sp500.csv", OHLCV_CSV)
        SKIP_DAILY = True
        print(f"Daily OHLCV: Drive'dan kopyalandi")
    if Path(f"{DRIVE_DIR}/data/intraday_1h_sp500.csv").exists():
        shutil.copy2(f"{DRIVE_DIR}/data/intraday_1h_sp500.csv", INTRADAY_CSV)
        SKIP_INTRADAY = True
        print(f"Intraday 1h: Drive'dan kopyalandi")
    if Path(f"{DRIVE_DIR}/data/news_sp500.json").exists():
        shutil.copy2(f"{DRIVE_DIR}/data/news_sp500.json", NEWS_JSON)
        SKIP_NEWS = True
        print(f"News: Drive'dan kopyalandi")

print(f"\nDaily: {'SKIP' if SKIP_DAILY else 'TOPLANACAK'}")
print(f"Intraday: {'SKIP' if SKIP_INTRADAY else 'TOPLANACAK'}")
print(f"News: {'SKIP' if SKIP_NEWS else 'TOPLANACAK'}")

In [ ]:
# S&P 500 tam liste (503 sembol)
from scripts.collect_sp500_full import SP500_ALL
print(f"S&P 500: {len(SP500_ALL)} sembol")

In [ ]:
# 4a. Daily OHLCV (503 sembol x 5 yil)
if not SKIP_DAILY:
    import yfinance as yf

    END_DATE = datetime.now().strftime("%Y-%m-%d")
    START_DATE = (datetime.now() - timedelta(days=365*5)).strftime("%Y-%m-%d")
    print(f"Daily OHLCV: {len(SP500_ALL)} symbols, {START_DATE} -> {END_DATE}")

    all_dfs = []
    failed = []
    batch_size = 50

    for i in range(0, len(SP500_ALL), batch_size):
        batch = SP500_ALL[i:i+batch_size]
        try:
            raw = yf.download(batch, start=START_DATE, end=END_DATE,
                              auto_adjust=True, progress=False, threads=True)
            for sym in batch:
                try:
                    sym_df = raw.xs(sym, level=1, axis=1).copy() if isinstance(raw.columns, pd.MultiIndex) else raw.copy()
                    sym_df = sym_df.dropna(subset=["Close"])
                    if len(sym_df) < 100:
                        failed.append(sym)
                        continue
                    sym_df = sym_df.reset_index()
                    sym_df.columns = [c.lower() for c in sym_df.columns]
                    sym_df["symbol"] = sym
                    sym_df = sym_df[["date", "open", "high", "low", "close", "volume", "symbol"]]
                    all_dfs.append(sym_df)
                except Exception:
                    failed.append(sym)
        except Exception as e:
            failed.extend(batch)
        if (i // batch_size + 1) % 2 == 0:
            print(f"  Batch {i//batch_size+1}/{(len(SP500_ALL)-1)//batch_size+1} done")
        time.sleep(1)

    ohlcv = pd.concat(all_dfs, ignore_index=True)
    ohlcv.to_csv(OHLCV_CSV, index=False)
    print(f"\nDaily OHLCV saved: {len(ohlcv)} rows, {ohlcv['symbol'].nunique()} symbols")
    if failed:
        print(f"Failed ({len(failed)}): {failed[:10]}...")
    if DRIVE_DIR:
        shutil.copy2(OHLCV_CSV, f"{DRIVE_DIR}/data/ohlcv_sp500.csv")
else:
    ohlcv = pd.read_csv(OHLCV_CSV)
    print(f"Daily OHLCV loaded: {len(ohlcv)} rows, {ohlcv['symbol'].nunique()} symbols")

In [ ]:
# 4b. Intraday 1h OHLCV (1h/4h horizon icin)
if COLLECT_INTRADAY and not SKIP_INTRADAY:
    import yfinance as yf

    print(f"Intraday 1h: {len(SP500_ALL)} symbols, son 2 yil")
    intra_dfs = []
    batch_size = 20

    for i in range(0, len(SP500_ALL), batch_size):
        batch = SP500_ALL[i:i+batch_size]
        try:
            raw = yf.download(batch, period="730d", interval="1h",
                              auto_adjust=True, progress=False, threads=True)
            for sym in batch:
                try:
                    sym_df = raw.xs(sym, level=1, axis=1).copy() if isinstance(raw.columns, pd.MultiIndex) else raw.copy()
                    sym_df = sym_df.dropna(subset=["Close"])
                    if len(sym_df) < 100:
                        continue
                    sym_df = sym_df.reset_index()
                    sym_df.columns = [c.lower().replace("datetime", "date") for c in sym_df.columns]
                    sym_df["symbol"] = sym
                    sym_df = sym_df[["date", "open", "high", "low", "close", "volume", "symbol"]]
                    intra_dfs.append(sym_df)
                except Exception:
                    pass
        except Exception:
            pass
        if (i // batch_size + 1) % 5 == 0:
            print(f"  Batch {i//batch_size+1}/{(len(SP500_ALL)-1)//batch_size+1}")
        time.sleep(2)

    if intra_dfs:
        intraday = pd.concat(intra_dfs, ignore_index=True)
        intraday.to_csv(INTRADAY_CSV, index=False)
        print(f"Intraday saved: {len(intraday)} rows, {intraday['symbol'].nunique()} symbols")
        if DRIVE_DIR:
            shutil.copy2(INTRADAY_CSV, f"{DRIVE_DIR}/data/intraday_1h_sp500.csv")
    else:
        print("Intraday: no data")
elif SKIP_INTRADAY:
    print(f"Intraday loaded from Drive")
else:
    print("Intraday collection skipped (COLLECT_INTRADAY=False)")

In [ ]:
# 4c. Finnhub Haberler (top N sembol x 5 yil)
if NEWS_TOP_N > 0 and not SKIP_NEWS:
    import finnhub

    client = finnhub.Client(api_key=FINNHUB_API_KEY)
    news_symbols = SP500_ALL[:NEWS_TOP_N]
    END_DATE = datetime.now().strftime("%Y-%m-%d")
    START_DATE = (datetime.now() - timedelta(days=365*5)).strftime("%Y-%m-%d")

    print(f"Collecting news for {len(news_symbols)} symbols: {START_DATE} -> {END_DATE}")
    print(f"Bu islem ~30-60 dk surebilir (rate limit: 60 req/min)")

    all_news = []
    start_dt = datetime.strptime(START_DATE, "%Y-%m-%d")
    end_dt = datetime.strptime(END_DATE, "%Y-%m-%d")

    for idx, sym in enumerate(news_symbols):
        sym_count = 0
        current = start_dt
        while current < end_dt:
            chunk_end = min(current + timedelta(days=90), end_dt)
            s = current.strftime("%Y-%m-%d")
            e = chunk_end.strftime("%Y-%m-%d")
            try:
                raw = client.company_news(sym, _from=s, to=e)
                if raw:
                    for item in raw:
                        all_news.append({
                            "headline": item.get("headline", ""),
                            "summary": item.get("summary", ""),
                            "source": item.get("source", ""),
                            "datetime": datetime.fromtimestamp(item.get("datetime", 0)).strftime("%Y-%m-%d %H:%M"),
                            "symbol": sym,
                        })
                        sym_count += 1
            except Exception as ex:
                if "429" in str(ex) or "Too Many" in str(ex):
                    time.sleep(30)
            current = chunk_end + timedelta(days=1)
            time.sleep(1.5)

        if (idx + 1) % 10 == 0:
            print(f"  [{idx+1}/{len(news_symbols)}] {sym}: {sym_count} articles (total: {len(all_news)})")

    import json
    Path(NEWS_JSON).write_text(json.dumps(all_news, ensure_ascii=False))
    print(f"\nNews saved: {len(all_news)} articles -> {NEWS_JSON}")
    if DRIVE_DIR:
        shutil.copy2(NEWS_JSON, f"{DRIVE_DIR}/data/news_sp500.json")
elif SKIP_NEWS:
    print(f"News loaded from Drive")
else:
    print("News collection skipped (NEWS_TOP_N=0)")

## 5. Dataset Build

**125 feature x 8 horizon:**

| Feature Grubu | Adet |
|---------------|------|
| Returns (DR, DR², rolling stats) | 6 |
| Shannon Entropy | 1 |
| FFT (top-32 frekans) | 64 |
| Wavelet decomposition | 20 |
| Teknik Indikatörler (RSI, MACD, BB, ...) | 30 |
| FinBERT Sentiment | 4 |

| Horizon | Periyot |
|---------|--------|
| 1d | 1 gün |
| 15d | 15 gün |
| 1m | 1 ay (~21 gün) |
| 3m | 3 ay (~63 gün) |
| 6m | 6 ay (~126 gün) |
| 1y | 1 yıl (~252 gün) |

In [ ]:
# Dataset build
if Path(DATASET_DIR).exists() and Path(f"{DATASET_DIR}/X.npy.dat").exists():
    print(f"Dataset zaten mevcut: {DATASET_DIR}")
    shape = np.load(f"{DATASET_DIR}/X_shape.npy")
    print(f"  Shape: ({int(shape[0])}, {int(shape[1])}, {int(shape[2])})")
    horizons = list(np.load(f"{DATASET_DIR}/horizon_names.npy"))
    print(f"  Horizons: {horizons}")
else:
    print("Building dataset...")
    print("503 sembol x 5 yil x 125 feature x 6 horizon")
    print("Tahmini sure: 30-60 dk\n")

    news_flag = "" if Path(NEWS_JSON).exists() else "--no-finbert"
    news_arg = f"--news {NEWS_JSON}" if Path(NEWS_JSON).exists() else ""

    !python scripts/build_dataset.py \
        --ohlcv {OHLCV_CSV} \
        {news_arg} \
        --out {DATASET_DIR} \
        --seq-len 60 \
        --threshold 0.01 \
        {news_flag}

    if Path(f"{DATASET_DIR}/X.npy.dat").exists():
        if DRIVE_DIR:
            if Path(f"{DRIVE_DIR}/data/sp500_dataset").exists():
                shutil.rmtree(f"{DRIVE_DIR}/data/sp500_dataset")
            shutil.copytree(DATASET_DIR, f"{DRIVE_DIR}/data/sp500_dataset")
            print("Dataset backed up to Drive.")
    else:
        print("Dataset build FAILED")

## 6. Model Training (GPU)

4 model + meta-learner, 8 horizon output head.

In [ ]:
from core.training.train_all import train_all_models

summary = train_all_models(
    market=MARKET,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device="cuda",
    data_path=DATASET_DIR,
)

In [ ]:
import json
print("\n" + "=" * 60)
print("TRAINING RESULTS")
print("=" * 60)
print(f"Market: {summary['market']}")
print(f"Horizons: {summary['horizons']}")
print(f"Total time: {summary['total_time_minutes']:.1f} min\n")
for name, result in summary["models"].items():
    print(f"  {name:20s} | val_loss={result['best_val_loss']:.4f} | epochs={result['epochs_trained']}")
print(f"\nModels: outputs/models/{MARKET}/")

## 7. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/logs/{MARKET}

## 8. Drive'a Kaydet

In [ ]:
if DRIVE_DIR:
    from pathlib import Path
    import shutil

    for src_name, dst_name in [(f"outputs/models/{MARKET}", f"models/{MARKET}"),
                                (f"outputs/logs/{MARKET}", f"logs/{MARKET}")]:
        src = Path(src_name)
        dst = Path(f"{DRIVE_DIR}/{dst_name}")
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            size = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file()) / 1e6
            print(f"{dst_name}: {size:.1f} MB")

    print("\nColab kapansa bile modeller Drive'da guvende!")
else:
    print("Drive yok. outputs/models/ klasorunu manuel indirin.")